# 📊 01 — Model Validation

**Purpose:** Load the trained YOLO model, run validation on test data, and visualize metrics (mAP, precision, recall, confusion matrix, prediction samples).

> Model path: `../models/livedet_best.pt`  
> Dataset config: `../data/data.yaml`

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────
import sys, os
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO

%matplotlib inline

# Paths
PROJECT_ROOT = Path("..").resolve()
MODEL_PATH = PROJECT_ROOT / "models" / "livedet_best.pt"
DATA_YAML = PROJECT_ROOT / "data" / "data.yaml"
OUTPUT_DIR = PROJECT_ROOT / "outputs" / "images"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Model path   : {MODEL_PATH}  {'✅ EXISTS' if MODEL_PATH.exists() else '❌ NOT FOUND'}")
print(f"Data YAML    : {DATA_YAML}  {'✅ EXISTS' if DATA_YAML.exists() else '❌ NOT FOUND'}")

In [ ]:
# ── Load Model ───────────────────────────────────────────────────────
model = YOLO(str(MODEL_PATH))
print("✅ Model loaded successfully")
print(f"   Classes: {model.names}")
model.info()

In [ ]:
# ── Run Validation ────────────────────────────────────────────────────
# Runs model.val() on the validation set defined in data.yaml
results = model.val(data=str(DATA_YAML), imgsz=640, verbose=True)

print("\n📊 Validation Metrics:")
print(f"   mAP@0.5       : {results.box.map50:.4f}")
print(f"   mAP@0.5:0.95  : {results.box.map:.4f}")
print(f"   Precision      : {results.box.mp:.4f}")
print(f"   Recall         : {results.box.mr:.4f}")

In [ ]:
# ── Per-Class Metrics Table ───────────────────────────────────────────
import pandas as pd

class_names = list(model.names.values())
per_class = []
for i, name in enumerate(class_names):
    per_class.append({
        "Class": name,
        "Precision": f"{results.box.p[i]:.4f}" if i < len(results.box.p) else "N/A",
        "Recall": f"{results.box.r[i]:.4f}" if i < len(results.box.r) else "N/A",
        "mAP@0.5": f"{results.box.ap50[i]:.4f}" if i < len(results.box.ap50) else "N/A",
        "mAP@0.5:0.95": f"{results.box.ap[i]:.4f}" if i < len(results.box.ap) else "N/A",
    })

df = pd.DataFrame(per_class)
print(df.to_string(index=False))

In [ ]:
# ── Visualize Predictions on Validation Samples ──────────────────────
import glob, yaml

# Find validation images from data.yaml
with open(str(DATA_YAML)) as f:
    data_cfg = yaml.safe_load(f)

val_dir = os.path.join(data_cfg.get("path", ""), data_cfg.get("val", "images/val"))
val_images = sorted(glob.glob(os.path.join(val_dir, "*.*")))[:6]

if not val_images:
    print("⚠️ No validation images found. Update val_dir path.")
else:
    cols = min(3, len(val_images))
    rows = (len(val_images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows))
    if rows == 1 and cols == 1:
        axes = np.array([axes])
    axes = axes.flatten()

    for idx, img_path in enumerate(val_images):
        img = cv2.imread(img_path)
        results_pred = model(img, conf=0.25, verbose=False)

        # Get annotated frame
        annotated = results_pred[0].plot()

        axes[idx].imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
        n_det = len(results_pred[0].boxes) if results_pred[0].boxes is not None else 0
        axes[idx].set_title(f"{os.path.basename(img_path)}\n{n_det} det.", fontsize=9)
        axes[idx].axis("off")

    for j in range(len(val_images), len(axes)):
        axes[j].axis("off")

    plt.suptitle("Validation Sample Predictions", fontsize=14, y=1.01)
    plt.tight_layout()
    plt.show()

In [ ]:
# ── Confusion Matrix ─────────────────────────────────────────────────
# The validation run saves a confusion matrix automatically.
cm_path = OUTPUT_DIR / "confusion_matrix_normalized.png"
cm_raw   = OUTPUT_DIR / "confusion_matrix.png"

for cm_file in [cm_path, cm_raw]:
    if cm_file.exists():
        img_cm = cv2.imread(str(cm_file))
        plt.figure(figsize=(8, 8))
        plt.imshow(cv2.cvtColor(img_cm, cv2.COLOR_BGR2RGB))
        plt.title(f"Confusion Matrix — {cm_file.name}", fontsize=13)
        plt.axis("off")
        plt.show()
    else:
        print(f"⚠️ {cm_file.name} not found – run the validation cell first.")

## 📝 Interpretation Notes

| Metric | What it means |
|--------|--------------|
| **mAP@0.5** | Mean Average Precision at IoU ≥ 0.5 — primary detection quality metric |
| **mAP@0.5:0.95** | Averaged over IoU thresholds 0.5 → 0.95 — stricter benchmark |
| **Precision** | Of everything the model flagged, how many were real defects? |
| **Recall** | Of all real defects, how many did the model find? |

### ✅ Good Results
- **mAP@0.5 ≥ 0.70** → Model is usable for production
- **Precision ≥ 0.80** → Few false positives
- **Recall ≥ 0.70** → Catches most defects

### ⚠️ If Results Are Poor
1. Increase training epochs or use a larger model (YOLOv8m → YOLOv8l)
2. Augment the dataset (flip, rotate, brightness jitter)
3. Check for class imbalance in the dataset